# Voice Guard v2 training

This notebook creates a transcript/prompt-disjoint v2 protocol, builds fresh frozen-feature caches, and compares model heads using validation only. The upstream dataset does not expose speaker or generator IDs, so the resulting claim is prompt-disjoint rather than unseen-speaker or unseen-generator performance. Enable a Kaggle GPU before running.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git"
REPO_DIR = Path("/kaggle/working/indic-fake-voice-detection")
WORK_DIR = Path("/kaggle/working/voice_guard_v2")
VOICE_GUARD_REF = os.environ.get("VOICE_GUARD_REF", "main").strip() or "main"
WORK_DIR.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", VOICE_GUARD_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-training.txt")], check=True)
RESOLVED_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", RESOLVED_SHA)
subprocess.run([sys.executable, "-c", "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], check=True)


## 1. Export metadata and freeze v2

This pass removes the audio column before streaming, so it reads metadata without decoding waveforms. The split audit is cryptographically bound to the generated split.

In [ ]:
METADATA = WORK_DIR / "metadata_v2.jsonl"
SPLIT = WORK_DIR / "split_ids_v2.json"
subprocess.run([sys.executable, str(REPO_DIR / "data_pipeline/export_hf_metadata.py"), "--output", str(METADATA)], check=True)
subprocess.run([sys.executable, str(REPO_DIR / "data_pipeline/build_group_safe_split.py"), str(METADATA), "--group-columns", "prompt_id", "--output", str(SPLIT)], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, str(REPO_DIR / "data_pipeline/leakage_audit.py"), "--split-file", str(SPLIT)], cwd=REPO_DIR, check=True)


## 2. Validation-only ablations

`smoke` is the safe default and is not reportable. After it completes, change `RUN_MODE` to `full`, restart the notebook from the top with a GPU, and run again. The full mode creates fresh v2 caches and trains all candidates without evaluating the test split.

In [ ]:
RUN_MODE = os.environ.get("VOICE_GUARD_TRAIN_MODE", "smoke").lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("VOICE_GUARD_TRAIN_MODE must be 'smoke' or 'full'.")
epochs = 2 if RUN_MODE == "smoke" else 15
max_clips = 400 if RUN_MODE == "smoke" else None
CACHE = WORK_DIR / ("feature_cache_smoke" if RUN_MODE == "smoke" else "feature_cache_full")
CHECKPOINTS = WORK_DIR / ("experiments_smoke" if RUN_MODE == "smoke" else "experiments_full")
RESULTS = WORK_DIR / f"validation_ablation_results_{RUN_MODE}.json"
command = [sys.executable, str(REPO_DIR / "run_validation_ablations.py"), "--split-file", str(SPLIT), "--metadata-file", str(METADATA), "--cache-dir", str(CACHE), "--checkpoint-root", str(CHECKPOINTS), "--output", str(RESULTS), "--epochs", str(epochs)]
if max_clips is not None:
    command += ["--max-clips-per-split", str(max_clips)]
print("Mode:", RUN_MODE, "| Results:", RESULTS)
subprocess.run(command, cwd=REPO_DIR, check=True)


## 3. Final-test gate

Do not evaluate test during model selection. First review the full validation ranking and robustness evidence, then set `VOICE_GUARD_LOCKED_CHECKPOINT` to the selected `best_model.pt`. Set `VOICE_GUARD_RUN_FINAL_TEST=1` only when the choice is final, and run this gate once.

In [ ]:
RUN_FINAL_TEST = os.environ.get("VOICE_GUARD_RUN_FINAL_TEST", "0").strip() == "1"
LOCKED_CHECKPOINT_TEXT = os.environ.get("VOICE_GUARD_LOCKED_CHECKPOINT", "").strip()
if RUN_FINAL_TEST:
    if RUN_MODE != "full":
        raise RuntimeError("Final test is disabled for smoke mode.")
    if not LOCKED_CHECKPOINT_TEXT:
        raise RuntimeError("Set VOICE_GUARD_LOCKED_CHECKPOINT to the selected best_model.pt first.")
    LOCKED_CHECKPOINT = Path(LOCKED_CHECKPOINT_TEXT).expanduser().resolve()
    if LOCKED_CHECKPOINT.name != "best_model.pt" or not LOCKED_CHECKPOINT.exists():
        raise FileNotFoundError("VOICE_GUARD_LOCKED_CHECKPOINT must point to an existing best_model.pt.")
    FINAL_CHECKPOINT_DIR = LOCKED_CHECKPOINT.parent
    if not (FINAL_CHECKPOINT_DIR / "step6_summary.json").exists():
        raise FileNotFoundError("The locked checkpoint directory also needs its matching step6_summary.json.")
    subprocess.run([sys.executable, str(REPO_DIR / "evaluate_test_split.py"), "--checkpoint-dir", str(FINAL_CHECKPOINT_DIR), "--cache-dir", str(CACHE), "--repo-dir", str(WORK_DIR / "final_test_artifacts"), "--split-file", str(SPLIT), "--metadata-file", str(METADATA)], cwd=REPO_DIR, check=True)
else:
    print("Final test remains locked. This is correct during model selection.")
